# GPS code point positioning — how we actually solve the least squares

**Station DYNG, 2024-01-01; GPS code observations only.**
We have already learned how the individual physical corrections work in
[`../gps_code_model_analysis.ipynb`](../gps_code_model_analysis.ipynb).
Now we focus on the numerical heart of our actual
[`gps_code_positioning.ipynb`](gps_code_positioning.ipynb):

- What are the **unknown parameters** and how are they ordered?
- What is one row of the **Jacobian/design matrix**?
- How do per-epoch **receiver clocks** enter that row?
- How do the **two-hour piecewise ZWD knots** enter? (Our main topic.)
- What are the observation vector, computed values, misclosures, weights
  and **post-fit residuals**?
- Why do we **iterate**, when do we stop, and how do we detect gross errors?
- How do we interpret RMS, $\hat\sigma_0$, formal parameter errors and
  residual patterns?

We will construct the system explicitly using NumPy, then compare it
numerically with `dsoclasses.gnss.code_zwd.solve_code_zwd()`.
The notebook is **not** a general least-squares framework; its matrix and
parameters are specific to our dual-frequency GPS **code-only** exercise.


## 1. Prepare exactly the code observations from our positioning exercise

These preparation cells are copied from the working GPS positioning notebook:
same RINEX, SP3, Bias-SINEX, ANTEX, solid Earth tides, VMF3, elevation
cutoff and correction signs. We sample every **15 minutes** here so the
matrices are easier to inspect in class; the original notebook samples
more frequently. We do **not** use carrier phase observations.

After these cells, `raw_obs` contains dictionaries that already hold one
**corrected ionosphere-free code observation** and the associated geometric
and atmospheric metadata. We study the least squares, not file parsing.


In [ ]:
from os.path import join

# This notebook lives in JupyterLab/gnss/beta/.
data_path = "../../../data"
rinex_path = join(data_path, "DYNG00GRC_R_20240010000_01D_30S_MO.rnx")
sp3_path = join(data_path, "COD0MGXFIN_20240010000_01D_05M_ORB.SP3")
vmf3_path = join(data_path, "y2024.vmf3_g")
bias_path = join(data_path, "COD0MGXFIN_20240010000_01D_01D_OSB.BIA")
atx_path = join(data_path, "igs20.atx")

# Keep these toggles visible to students: each has a physical meaning.
EVERY_SEC = 900
ELEVATION_CUTOFF_DEG = 3.0
ZWD_EVERY_HOURS = 2.0
SIGMA_CODE_M = 5.0
APPLY_SOLID_EARTH_TIDES = True
APPLY_SATELLITE_NADIR_PCO = True
EXCLUDE_SATELLITES = set()


In [ ]:
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from dsoclasses.orbits.interpolator import Sp3Interpolator
from dsoclasses.rinex.gnss.rinex import GnssRinex, fetch
from dsoclasses.gnss import algorithms as alg
from dsoclasses.gnss import systems as gs
from dsoclasses.gnss.atx import Atx
from dsoclasses.gnss.biasx import Bsinex
from dsoclasses.gnss.code_zwd import solve_code_zwd
from dsoclasses.troposphere.vmf3 import SiteVmf3
from dsoclasses.geodesy import transformations
from dsoclasses.geodesy.earth_models import solid_earth_tide_xyz
from dsoclasses.time.astropy_time import to_astropy_time


In [ ]:
# A GNSS satellite identifier always includes its constellation letter.
SYSTEM = 'G'
FREQ1_ATX, FREQ2_ATX = 'G01', 'G02'
CODE1 = ('C1W', 'C1C', 'C1X', 'C1L')
CODE2 = ('C2W', 'C2C', 'C2D', 'C2P', 'C2X', 'C2S', 'C2L')
ALLOW_MISSING_SATELLITE_OSB = False

def satellite_frequencies(sat):
    # MHz are sufficient: only ratios of squared frequencies are used.
    return gs.GPS_L1_FREQ, gs.GPS_L2_FREQ


In [ ]:
rnx = GnssRinex(rinex_path)
intrp = Sp3Interpolator.from_sp3(
    sp3_path, [SYSTEM], interval_in_sec=2400,
    min_data_pts=8, stencil_pts=8, itype='Barycentric',
)

# The source readers deliberately do NOT convert between time systems.
# Reject mismatched time tags instead of accidentally treating UTC/GLO as GPS.
rinex_scale = (rnx.time_sys or '').strip().upper()
sp3_scale = intrp.time_system().strip().upper()
print('RINEX/SP3 time scales:', rinex_scale, '/', sp3_scale)
if rinex_scale != sp3_scale:
    raise ValueError('RINEX and SP3 have different time scales; convert the '
                     'epoch labels explicitly before positioning')
print('RINEX first epoch as UTC (Astropy):',
      to_astropy_time(rnx.time_first_obs, rinex_scale).utc.isot)

marker_xyz = np.array(rnx.approx_cartesian(), dtype=float)
latitude, longitude, height_m = transformations.car2ell(*marker_xyz)
site_id = rnx.marker_name[:4].upper()  # VMF3 site-wise stations use 4-char IDs.

vmf = SiteVmf3(vmf3_path, site_list=[site_id])
osb = Bsinex(bias_path)
antex = Atx(atx_path)
receiver_antenna = antex.get_noazi(rnx.antenna_type,
                                    [FREQ1_ATX, FREQ2_ATX])

# RINEX header ANTENNA: DELTA H/E/N = marker -> ARP, metres.
if not all(hasattr(rnx, key) for key in ('dh', 'de', 'dn')):
    raise ValueError('RINEX lacks ANTENNA: DELTA H/E/N; marker/ARP offset unknown')
marker_to_arp_ecef = transformations.geodetic2lvlh(latitude, longitude) @ np.array(
    [rnx.de, rnx.dn, rnx.dh], dtype=float
)

print('Station:', rnx.marker_name, ' / antenna:', rnx.antenna_type)
print('RINEX observables:', rnx.obscodes.get(SYSTEM, []))
print('Marker -> ARP offset [m]:', marker_to_arp_ecef)


In [ ]:
# Collect only ordinary code observations with complete required corrections.
# Expected missing data are counted; programming mistakes remain visible.
raw_obs = []
skip_reasons = Counter()
missing_osb = Counter()
num_epochs = 0
last_used_astropy_time = None
first_epoch_astropy = to_astropy_time(rnx.time_first_obs, rinex_scale)
knot_spacing_s = ZWD_EVERY_HOURS * 3600
elevation_cutoff = np.deg2rad(ELEVATION_CUTOFF_DEG)


def code_osb_metres(sat, code, t):
    try:
        return osb.code_bias_m(sat, code, t)
    except (LookupError, ValueError) as exc:
        if not ALLOW_MISSING_SATELLITE_OSB:
            raise
        # Some GLONASS products have no OSB for particular FDMA signals.
        # In that case our demonstrator deliberately uses zero but REPORTS it.
        missing_osb[(sat, code)] += 1
        return 0.0


for block in rnx:
    t = block.t()
    astro_t = to_astropy_time(t, rinex_scale)
    elapsed_s = (astro_t - first_epoch_astropy).to_value('s')
    if (last_used_astropy_time is not None and
            (astro_t - last_used_astropy_time).to_value('s') < EVERY_SEC):
        continue
    if elapsed_s < 0:
        continue

    # One solid-Earth-tide displacement per epoch, not per satellite.
    tide_xyz = (solid_earth_tide_xyz(marker_xyz, t, rinex_scale)
                if APPLY_SOLID_EARTH_TIDES else np.zeros(3))
    station_displacement = marker_to_arp_ecef + tide_xyz
    approx_arp = marker_xyz + station_displacement

    # Linear interpolation between the surrounding two-hour ZWD knots.
    k, remainder_s = divmod(elapsed_s, knot_spacing_s)
    alpha = remainder_s / knot_spacing_s
    zwd_basis = [(int(k), 1.0 - alpha), (int(k) + 1, alpha)]
    candidates = []

    for sat, obs in block.filter_satellite_system(SYSTEM, False):
        if sat in EXCLUDE_SATELLITES:
            continue
        try:
            first, code1 = fetch(obs, *CODE1)
            second, code2 = fetch(obs, *CODE2)
            f1, f2 = satellite_frequencies(sat)
            divisor = f1*f1 - f2*f2

            p1 = float(first['value']) - code_osb_metres(sat, code1, t)
            p2 = float(second['value']) - code_osb_metres(sat, code2, t)

            xs, ys, zs, satellite_clock_s, tau = alg.sat_at_emission_time(
                approx_arp, t, intrp, sat)
            satellite_xyz = np.array([xs, ys, zs])
            _, az, el = transformations.azele(satellite_xyz, approx_arp)
            if el < elevation_cutoff:
                skip_reasons['Below elevation cutoff'] += 1
                continue

            # Receiver ANTEX PCO + NOAZI PCV: already in metres.
            p1 += receiver_antenna.range_correction_m(FREQ1_ATX, az, el)
            p2 += receiver_antenna.range_correction_m(FREQ2_ATX, az, el)

            if APPLY_SATELLITE_NADIR_PCO:
                # Approximation: satellite body +Z points to geocentre.
                # The X/Y PCO components and satellite PCV are not modeled.
                p1 += antex.satellite_nadir_range_correction_m(
                    sat, FREQ1_ATX, t, satellite_xyz, approx_arp,
                    travel_time_s=tau)
                p2 += antex.satellite_nadir_range_correction_m(
                    sat, FREQ2_ATX, t, satellite_xyz, approx_arp,
                    travel_time_s=tau)

            p_iono_free = (f1*f1*p1 - f2*f2*p2) / divisor
            meteo = vmf.vmf3(site_id, latitude, longitude, el, t)
            relativity = alg.relativistic_range_term(
                t, intrp, sat, travel_time_s=tau)
            sagnac = alg.sagnac_range_term(satellite_xyz, approx_arp)

            model_values = [p_iono_free, satellite_clock_s, relativity,
                            sagnac, meteo['zhd'], meteo['zwd'],
                            meteo['mfh'], meteo['mfw']]
            if not (np.all(np.isfinite(model_values)) and
                    np.all(np.isfinite(satellite_xyz))):
                raise ValueError('Non-finite code observation or model correction')

            candidates.append({
                't': t, 'sat': sat, 'p3': p_iono_free,
                'rsat': satellite_xyz, 'csat': satellite_clock_s,
                'el': el, 'dT': meteo, 'drel': relativity,
                'dsagnac': sagnac, 'zwd_basis': zwd_basis,
                'station_displacement_m': station_displacement,
            })
        except (LookupError, KeyError, ValueError, RuntimeError, ArithmeticError) as exc:
            skip_reasons[f'{type(exc).__name__}: {exc}'] += 1

    # A standalone clock epoch with too few satellites is poorly constrained.
    if len(candidates) < 4:
        skip_reasons['Epoch with fewer than four complete code satellites'] += 1
        continue
    for candidate in candidates:
        candidate['nepoch'] = num_epochs
        raw_obs.append(candidate)
    last_used_astropy_time = astro_t
    num_epochs += 1

print(f'Accepted {len(raw_obs)} code observations in {num_epochs} epochs')
for reason, count in skip_reasons.most_common(12):
    print(f'  skipped {count:5d}  {reason}')
if missing_osb:
    print('WARNING: missing satellite OSB values explicitly set to zero:')
    for key, count in missing_osb.most_common(10):
        print(f'  {key}: {count} observations')
if not raw_obs:
    raise RuntimeError('No valid code observations; check the skip summary, '
                       'signal codes, epoch scales and correction-file coverage')


## 2. Define the actual observation equation

For a code observation $i$ of satellite $s$ at reception epoch $k$,
the correction models produce

$$\ell_i=P^{corr}_{IF,i}+c\,\delta t_{s,i}.$$

The computed value at an assumed parameter vector $\boldsymbol\theta$ is

$$
f_i(\boldsymbol\theta)=
\underbrace{\left\|\mathbf r_{s,i}-
(\mathbf r_{marker}+\mathbf d_i)\right\|}_{\rho_i}
+\underbrace{b_k}_{c\delta t_{r,k}}
+\underbrace{ZHD_i\,m_{h,i}}_{\mathrm{known\ hydrostatic}}
+\underbrace{ZWD(t_i)\,m_{w,i}}_{\mathrm{estimated\ wet}}
+\underbrace{\Delta\rho_{rel,i}+\Delta\rho_{Sag,i}}_{\mathrm{known\ models}}.
$$

Here $\mathbf d_i$ includes the RINEX marker-to-ARP eccentricity **and**
solid Earth tides. The unknown receiver-clock parameter $b_k$ is **metres**
($b_k=c\delta t_{r,k}$), not seconds. ZWD is also in **metres**.

The *pre-fit misclosure* is $w_i=\ell_i-f_i(\boldsymbol\theta^{(0)})$;
the *post-fit residual* is $v_i=\ell_i-f_i(\hat{\boldsymbol\theta})$.
Both are in metres, but only $v_i$ follows the final parameter update.


In [ ]:
print(f'Observations: {len(raw_obs)} (each row is one GPS code measurement)')
print(f'Receiver-clock epochs: {num_epochs}')
one = raw_obs[0]
print('Example observation satellite:', one['sat'])
print('Corrected IF code P3 [m]:', one['p3'])
print('Satellite clock [s]:', one['csat'])
print('Known antenna/ground displacement [m]:', one['station_displacement_m'])
print('Mapping functions (hydrostatic/wet):', one['dT']['mfh'], one['dT']['mfw'])
print('ZWD interpolation weights:', one['zwd_basis'])
print('Clock epoch index:', one['nepoch'])

observed_m = np.array([o['p3'] + gs.C*o['csat'] for o in raw_obs])
assert observed_m.ndim == 1 and np.all(np.isfinite(observed_m))
print('Observed vector ℓ shape:', observed_m.shape)


## 3. Parameter vector: XYZ + **one clock per epoch** + ZWD knots

For $N_e$ accepted epochs and $N_z$ wet-delay knots, our parameter vector is

$$
\boldsymbol\theta=\left[\begin{array}{c}
X\\Y\\Z\\
b_0\\b_1\\\vdots\\b_{N_e-1}\\
ZWD_0\\ZWD_1\\\vdots\\ZWD_{N_z-1}
\end{array}\right]
\quad\in\mathbb R^{3+N_e+N_z}.
$$

The position has **three** global parameters. **Every accepted RINEX epoch
has its own clock parameter** in metres. This is essential: the receiver
clock is generally not the same at successive observation epochs.

The wet-delay parameters are **not one ZWD per observation or epoch**:
they are shared **two-hour knot values**. Each observation uses only its
two adjacent knots. That is what keeps the wet-delay model continuous.


In [ ]:
n_obs = len(raw_obs)
n_epochs = num_epochs
n_zwd = max(o['zwd_basis'][1][0] for o in raw_obs) + 1
n_params = 3 + n_epochs + n_zwd

# Source implementation: XYZ = RINEX approximate coordinates;
# receiver clocks = zero metres; each wet knot = mean VMF3 a priori ZWD.
prior_zwd = float(np.mean([o['dT']['zwd'] for o in raw_obs]))
x0 = np.concatenate([
    marker_xyz,
    np.zeros(n_epochs),
    np.full(n_zwd, prior_zwd),
])

print(f'Unknowns: 3 XYZ + {n_epochs} receiver clocks + {n_zwd} ZWD knots '
      f'= {n_params}')
print(f'Observations / unknowns: {n_obs} / {n_params}')
print('Parameter blocks (Python indexing):')
print('  XYZ:   x[0:3]')
print(f'  clock: x[3:{3+n_epochs}]')
print(f'  ZWD:   x[{3+n_epochs}:{n_params}]')

if n_obs <= n_params:
    raise ValueError('This subset is not overdetermined; reduce EVERY_SEC')
assert all(o['zwd_basis'][1][0] == o['zwd_basis'][0][0] + 1
           for o in raw_obs)
assert all(np.isclose(sum(a for _, a in o['zwd_basis']), 1.)
           for o in raw_obs)


## 4. **Piecewise-linear ZWD: the important construction**

Let the ZWD knot spacing be $\Delta T=2$ hours. Define

$$k=\left\lfloor\frac{t_i-t_0}{\Delta T}\right\rfloor,
\qquad \alpha=\frac{(t_i-t_0)-k\Delta T}{\Delta T},
\qquad 0\le\alpha<1.$$

Our model interpolates between **two unknown knot values**:

$$\boxed{ZWD(t_i)=(1-\alpha)\,ZWD_k+\alpha\,ZWD_{k+1}}.$$

Therefore one observation's **slant wet delay** is

$$T_{wet,i}=m_{w,i}\big[(1-\alpha)ZWD_k+\alpha ZWD_{k+1}\big].$$

Its Jacobian has exactly **two nonzero ZWD entries**:

$$\frac{\partial f_i}{\partial ZWD_k}=m_{w,i}(1-\alpha),\qquad
\frac{\partial f_i}{\partial ZWD_{k+1}}=m_{w,i}\alpha.$$

**Example:** 1 hour after the earlier two-hour knot, $\alpha=0.5$.
If $ZWD_k=0.12$ m, $ZWD_{k+1}=0.16$ m, and $m_w=2.3$,
then $T_{wet}=2.3(0.5\cdot0.12+0.5\cdot0.16)=0.322$ m.
Neither ZWD value is a fixed correction: **both are estimated jointly
with the station coordinates and clocks**.


In [ ]:
def wet_knot_terms(obs, n_epochs):
    """Return (column, Jacobian value) for exactly two ZWD knots."""
    mw = float(obs['dT']['mfw'])
    return [(3+n_epochs+index, mw*weight)
            for index, weight in obs['zwd_basis']]

example = raw_obs[0]
print('Real observation ZWD interpolation:', example['zwd_basis'])
print('Slant wet mapping factor:', example['dT']['mfw'])
print('The TWO columns/nonzero coefficients are:',
      wet_knot_terms(example, n_epochs))

# A hand-verifiable 1-hour midpoint example, independent of the real epoch.
alpha_demo = .5
z0_demo, z1_demo, mw_demo = .12, .16, 2.3
wet_demo = mw_demo*((1-alpha_demo)*z0_demo+alpha_demo*z1_demo)
print(f'Slant wet delay at the midpoint = {wet_demo:.3f} m')
assert np.isclose(wet_demo, .322)

# Plot the actual basis weights from the real RINEX observation epochs.
astro0 = to_astropy_time(rnx.time_first_obs, rinex_scale)
obs_times_h = np.array([(to_astropy_time(o['t'], rinex_scale)-astro0).to_value('h')
                        for o in raw_obs])
knots0 = np.array([o['zwd_basis'][0][0] for o in raw_obs])
w0 = np.array([o['zwd_basis'][0][1] for o in raw_obs])
w1 = np.array([o['zwd_basis'][1][1] for o in raw_obs])

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(obs_times_h, w0, s=8, label='Earlier knot weight: 1−α')
ax.scatter(obs_times_h, w1, s=8, label='Later knot weight: α')
for hour in np.arange(0, np.ceil(obs_times_h.max()/2)*2+1, 2):
    ax.axvline(hour, color='grey', lw=.5, alpha=.45)
ax.set(xlabel='Hours since first observation', ylabel='Interpolation weight',
       title='Real RINEX epochs: two-hour ZWD basis (weights always sum to 1)')
ax.legend()
ax.grid(alpha=.25)
plt.show()
assert np.allclose(w0+w1, 1.)


## 5. Build one row of the Jacobian (design matrix)

The unknown position enters through geometric range:

$$\rho_i=\|\mathbf r_{marker}+\mathbf d_i-\mathbf r_{s,i}\|.$$

Let $\mathbf q_i=\mathbf r_{marker}+\mathbf d_i-\mathbf r_{s,i}$.
Then the XYZ partial derivatives of the **computed** value are

$$\boxed{\frac{\partial f_i}{\partial (X,Y,Z)}=
\frac{\mathbf q_i^T}{\rho_i}}.$$

For the clock at this observation's epoch $k$:

$$\frac{\partial f_i}{\partial b_k}=1,
\qquad\frac{\partial f_i}{\partial b_{j\ne k}}=0.$$

The two adjacent ZWD coefficients come from the previous section.
Thus one Jacobian row has at most **six nonzero entries**:
3 XYZ, 1 receiver-clock column and 2 ZWD-knot columns.

Writing the observation difference as $\mathbf w=\boldsymbol\ell-
\mathbf f(\boldsymbol\theta)$, the **linearized** equation is

$$\boxed{\mathbf A\,\Delta\boldsymbol\theta \simeq \mathbf w},
\qquad A_{ij}=\frac{\partial f_i}{\partial\theta_j}.$$

Note the sign: $\mathbf A$ differentiates the **computed** value $f$.
The derivative of residual $\ell-f$ itself is $-\mathbf A$.


In [ ]:
def compute_and_jacobian(parameters, observations, n_epochs, n_zwd):
    """Explicit, dense code/ZWD prediction and design matrix for class."""
    n = len(observations)
    p = 3 + n_epochs + n_zwd
    calculated = np.zeros(n)
    A = np.zeros((n, p))
    for i, obs in enumerate(observations):
        sta = parameters[:3] + np.asarray(obs['station_displacement_m'], dtype=float)
        diff = sta - np.asarray(obs['rsat'], dtype=float)
        rho = float(np.linalg.norm(diff))
        if not np.isfinite(rho) or rho <= 0:
            raise ValueError(f'Invalid geometric range in row {i}')

        k = obs['nepoch']
        calculated[i] = (rho + parameters[3+k]
                         + obs['dT']['zhd']*obs['dT']['mfh']
                         + obs['drel'] + obs['dsagnac'])

        A[i, :3] = diff/rho                  # X, Y, Z
        A[i, 3+k] = 1.0                    # THIS epoch's clock
        for knot_col, coefficient in wet_knot_terms(obs, n_epochs):
            calculated[i] += coefficient*parameters[knot_col]
            A[i, knot_col] = coefficient   # only TWO ZWD columns

    return calculated, A

computed0, A0 = compute_and_jacobian(x0, raw_obs, n_epochs, n_zwd)
misclosure0 = observed_m-computed0
print('Computed vector f(x0):', computed0.shape)
print('Pre-fit misclosure ℓ - f(x0):', misclosure0.shape)
print('Jacobian A:', A0.shape)

# Choose a time BETWEEN knots, so both ZWD partials are nonzero.
row_index = next(i for i, obs in enumerate(raw_obs)
                 if all(w > 0 for _, w in obs['zwd_basis']))
nonzero = np.flatnonzero(np.abs(A0[row_index]) > 1e-12)
print('\nNonzero columns of one REAL observation row:')
for col in nonzero:
    name = ('XYZ' if col < 3 else
            'clock' if col < 3+n_epochs else 'ZWD knot')
    print(f'  col {col:3d} [{name:<8s}]: {A0[row_index, col]:+.6f}')
assert len(nonzero) == 6
assert np.isclose(A0[row_index, 3+raw_obs[row_index]['nepoch']], 1.)


In [ ]:
# One picture captures the sparsity: every satellite sees XYZ, but only
# one clock epoch and two neighboring wet-delay knots.
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(np.abs(A0[:min(150,n_obs)]) > 1e-12, aspect='auto',
          interpolation='nearest', cmap='Greys')
ax.axvline(2.5, c='tab:red', lw=1, label='XYZ / clocks')
ax.axvline(2.5+n_epochs, c='tab:blue', lw=1, label='clocks / ZWD')
ax.set(xlabel='Parameter column', ylabel='Observation row',
       title='Nonzero pattern of the GPS code Jacobian')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'Dense A dimensions: {n_obs} × {n_params}')
print('Nearly all clock and ZWD entries are zero, but dense NumPy is'
      ' intentional for this lesson.')


## 6. The weight matrix $\mathbf W$

We do **not** assume every code measurement has equal precision. Our
positioning notebook uses a simple elevation weighting model:

$$\sigma_i=\frac{\sigma_{code}}{\sin(E_i)},\qquad
\sigma_{code}=5\;\mathrm m,\qquad
\mathbf W=\mathrm{diag}\left(\frac{1}{\sigma_i^2}\right).$$

Thus a low-elevation observation has a larger assumed uncertainty and
receives **less** weight. These are *a priori* standard deviations,
not empirical noise estimates for the particular observation.

Weighted least squares minimizes

$$\sum_i w_i\,\big(\ell_i-f_i(\boldsymbol\theta)\big)^2.
$$

The linearized normal equations can be written

$$\mathbf A^T\mathbf W\mathbf A\,\Delta\boldsymbol\theta
=\mathbf A^T\mathbf W\mathbf w.$$

For numerical robustness we solve the **whitened** system
$\mathbf A_w\Delta\boldsymbol\theta\simeq\mathbf w_w$ using QR/SVD
via `np.linalg.lstsq`—**not** by explicitly inverting $A^TWA$:

$$\mathbf A_w=\mathbf W^{1/2}\mathbf A,\qquad
\mathbf w_w=\mathbf W^{1/2}\mathbf w.$$


In [ ]:
el_rad = np.array([o['el'] for o in raw_obs], dtype=float)
sigma_i = SIGMA_CODE_M/np.sin(el_rad)
weights_prior = 1.0/sigma_i**2

print('First five elevations [deg]:', np.round(np.rad2deg(el_rad[:5]), 2))
print('First five a priori sigmas [m]:', np.round(sigma_i[:5], 2))
print('First 5×5 block of diagonal W [1/m²]:')
print(np.diag(weights_prior[:5]))
assert np.all(weights_prior > 0)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].scatter(np.rad2deg(el_rad), sigma_i, s=8)
ax[0].set(xlabel='Elevation [deg]', ylabel='Assumed code σ [m]')
ax[1].scatter(np.rad2deg(el_rad), weights_prior, s=8)
ax[1].set(xlabel='Elevation [deg]', ylabel='Weight [1/m²]')
for a in ax: a.grid(alpha=.25)
plt.tight_layout()
plt.show()


## 7. One **weighted linearized step**

Starting from the initial marker XYZ, zero receiver-clock ranges, and
VMF3 a priori wet delays, compute

$$\mathbf w^{(0)}=\boldsymbol\ell-\mathbf f(\boldsymbol\theta^{(0)}),
\qquad
\Delta\boldsymbol\theta^{(0)}
=\arg\min_{\Delta\theta}\|\mathbf W^{1/2}
(\mathbf A\Delta\boldsymbol\theta-\mathbf w)\|^2.$$

Then **update** $\boldsymbol\theta^{(1)}=
\boldsymbol\theta^{(0)}+\Delta\boldsymbol\theta^{(0)}$.
The geometry depends on XYZ, so one linearized solution is not generally
final. We must rebuild $A$ and the modeled values at the new position.


In [ ]:
root_w = np.sqrt(weights_prior)
Aw0 = A0*root_w[:, None]
ww0 = misclosure0*root_w
dx0, _, rank0, singular0 = np.linalg.lstsq(Aw0, ww0, rcond=None)
if rank0 != n_params:
    raise ValueError(f'Initial design is rank deficient: {rank0}/{n_params}')

x1 = x0+dx0
calculated1, A1 = compute_and_jacobian(x1, raw_obs, n_epochs, n_zwd)
residual1 = observed_m-calculated1  # recomputed at UPDATED unknowns

print('Design matrix rank:', rank0, '/', n_params)
print('First XYZ increment [m]:', dx0[:3])
print('Largest receiver clock increment [m]:',
      np.max(np.abs(dx0[3:3+n_epochs])))
print('Largest ZWD knot increment [m]:',
      np.max(np.abs(dx0[3+n_epochs:])))
print('Before weighted RMS [dimensionless]:',
      np.sqrt(np.mean((misclosure0/sigma_i)**2)))
print('After first update weighted RMS [dimensionless]:',
      np.sqrt(np.mean((residual1/sigma_i)**2)))
assert np.all(np.isfinite(dx0))


## 8. Iterate, reject gross errors, **stop**

Our implementation is a *Gauss–Newton* iteration: we repeatedly calculate
the nonlinear prediction, linearize it, solve the weighted increment and
update the unknowns.

At each iteration:

1. Compute $\mathbf w=\ell-f(\boldsymbol\theta)$ and $A$.
2. Solve $A_w\Delta\theta\simeq w_w$ with `np.linalg.lstsq`.
3. Set $\boldsymbol\theta\leftarrow\boldsymbol\theta+\Delta\theta$.
4. **Recompute** the true nonlinear *post-update* residuals.
5. Reject an active observation when
   $|v_i|/\sigma_i>4$ (at most three rejection rounds), then **re-solve**.
6. Otherwise stop when both conditions hold:

$$\|\Delta\mathbf r_{XYZ}\|<10^{-4}\;\mathrm m,
\qquad\max_{j\ge3}|\Delta\theta_j|<10^{-4}\;\mathrm m.$$

We also impose an upper limit of **10 iterations**. If the design is rank
deficient, rejections leave too few observations, or convergence fails,
we raise an error; a plausible-looking coordinate is not a successful
solution. These thresholds match `solve_code_zwd` in `dsoclasses`.


In [ ]:
def iterate_code_positioning(observed, observations, initial, prior_sigma,
                             n_epochs, n_zwd,
                             max_iterations=10, max_rejections=3,
                             rejection_sigma=4.0, tolerance_m=1e-4):
    """Our code-specific weighted, iterative LS; readable over optimized."""
    x = initial.copy()
    W_diag = 1.0/prior_sigma**2
    history = []
    rejection_rounds = 0
    n_parameters = len(x)

    for iteration in range(1, max_iterations+1):
        calculated, A = compute_and_jacobian(x, observations, n_epochs, n_zwd)
        w = observed-calculated             # PRE-update misclosure
        active = W_diag > 0
        if np.count_nonzero(active) <= n_parameters:
            raise ValueError('Too few active observations for this LS system')

        sqrt_w = np.sqrt(W_diag[active])
        Aw = A[active]*sqrt_w[:, None]
        ww = w[active]*sqrt_w
        dx, _, rank, _ = np.linalg.lstsq(Aw, ww, rcond=None)
        if rank != n_parameters:
            raise ValueError(f'Rank-deficient Jacobian {rank}/{n_parameters}')

        x += dx
        new_calculated, _ = compute_and_jacobian(x, observations, n_epochs, n_zwd)
        vpost = observed-new_calculated    # POST-update nonlinear residual!
        outliers = active & (np.abs(vpost)/prior_sigma > rejection_sigma)

        history.append(dict(iteration=iteration,
                            xyz_step_m=float(np.linalg.norm(dx[:3])),
                            max_other_step_m=float(np.max(np.abs(dx[3:]))),
                            postfit_rms_m=float(np.sqrt(np.mean(vpost[active]**2))),
                            rejected_now=int(outliers.sum())))

        if outliers.any():
            if rejection_rounds >= max_rejections:
                raise RuntimeError('Too many gross-error rejection rounds')
            W_diag[outliers] = 0.0
            rejection_rounds += 1
            continue                     # MUST solve again with new weights

        if (np.linalg.norm(dx[:3]) < tolerance_m
                and np.max(np.abs(dx[3:])) < tolerance_m):
            break
    else:
        raise RuntimeError('Code-positioning least squares did not converge')

    # This residual vector corresponds to the FINAL parameters, including
    # rejected observations (which have zero weight in diagnostics).
    final_calculated, final_A = compute_and_jacobian(x, observations, n_epochs, n_zwd)
    final_v = observed-final_calculated
    dof = np.count_nonzero(W_diag) - n_parameters
    if dof <= 0:
        raise ValueError('No positive degrees of freedom')
    sigma_hat0 = np.sqrt(np.sum(W_diag*final_v**2)/dof)
    return x, final_v, W_diag, final_A, sigma_hat0, history

x, vpost, w_final, A_final, sigma_hat0, history = iterate_code_positioning(
    observed_m, raw_obs, x0, sigma_i, n_epochs, n_zwd)

print('Iteration  | ΔXYZ norm [m]  | max other Δ [m] | postfit RMS [m] | new rejects')
for h in history:
    print(f'{h["iteration"]:9d}  {h["xyz_step_m"]:15.6g}  '
          f'{h["max_other_step_m"]:15.6g}  '
          f'{h["postfit_rms_m"]:15.6g}  {h["rejected_now"]:5d}')
print('Final estimated marker XYZ [m]:', x[:3])
print('RINEX approximate XYZ [m]:', marker_xyz)
print('Rejected rows:', np.count_nonzero(w_final == 0))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
steps = np.array([h['iteration'] for h in history])
ax[0].semilogy(steps, [h['xyz_step_m'] for h in history], 'o-')
ax[0].axhline(1e-4, ls='--', c='tab:red', label='XYZ stopping tolerance')
ax[0].set(xlabel='Iteration', ylabel='||ΔXYZ|| [m]')
ax[0].legend()
ax[1].semilogy(steps, [h['max_other_step_m'] for h in history], 'o-')
ax[1].axhline(1e-4, ls='--', c='tab:red', label='Clock/ZWD tolerance')
ax[1].set(xlabel='Iteration', ylabel='max |Δclock or ΔZWD| [m]')
ax[1].legend()
for a in ax: a.grid(alpha=.25)
plt.tight_layout()
plt.show()


## 9. Verify our implementation against the **source-code solver**

We did not need to change `dsoclasses/gnss/code_zwd.py` to teach its
mathematics. The reference implementation already performs the same
nonlinear weighted solution, with convergence checks and gross-error
rejection. The two independent code paths should agree to numerical
precision; if they disagree, we must investigate, not quietly accept
one answer.


In [ ]:
reference = solve_code_zwd(
    raw_obs, marker_xyz, num_epochs, sigma0=SIGMA_CODE_M,
    max_iterations=10, max_rejections=3, rejection_sigma=4.0,
)
parameter_discrepancy = np.max(np.abs(x-reference.parameters))
residual_discrepancy = np.max(np.abs(vpost-reference.residuals))
print('Largest parameter discrepancy [m]:', parameter_discrepancy)
print('Largest postfit residual discrepancy [m]:', residual_discrepancy)
print('Same active/rejected rows:',
      np.array_equal(w_final > 0, reference.weights > 0))
print('σ̂₀, notebook / dsoclasses:', sigma_hat0, '/', reference.sigma_hat0)
assert parameter_discrepancy < 1e-5
assert residual_discrepancy < 1e-5
assert np.array_equal(w_final > 0, reference.weights > 0)


## 10. Post-fit residual diagnostics: RMS, standard deviation, $\hat\sigma_0$

Our **final residual vector** is

$$\mathbf v=\boldsymbol\ell-\mathbf f(\hat{\boldsymbol\theta}).$$

For active observations:

$$\mathrm{RMS}=\sqrt{\frac{1}{n}\sum_i v_i^2},\qquad
\bar v=\frac1n\sum_i v_i,\qquad
s_v=\sqrt{\frac{1}{n-1}\sum_i(v_i-\bar v)^2}.$$

The **unit-weight standard deviation** is *different*:

$$\boxed{\hat\sigma_0=\sqrt{\frac{\mathbf v^T\mathbf W\mathbf v}
{n_{active}-n_{parameters}}}}.$$

$\hat\sigma_0$ is **dimensionless** because our $W$ is $1/\mathrm m^2$.
It compares residual scatter with the assumed elevation-dependent code
standard deviations. It is **not a coordinate uncertainty**, and a small
$\hat\sigma_0$ does not independently prove that the station coordinates
are correct. The RMS is in **metres**.

We also inspect standardized residuals $v_i/\sigma_i$ and their
variation with elevation. Outlier rejection uses the *a priori*
$\sigma_i$, not an inflated post-fit estimate.


In [ ]:
active = w_final > 0
good_resid = vpost[active]
n_active = int(np.count_nonzero(active))
dof = n_active-n_params
residual_column = vpost.reshape(-1, 1)

print('Residual COLUMN vector shape:', residual_column.shape)
print(f'Active observations: {n_active}; parameters: {n_params}; '
      f'degrees of freedom: {dof}')
print(f'Mean residual [m]: {good_resid.mean():+.4f}')
print(f'Median residual [m]: {np.median(good_resid):+.4f}')
print(f'Residual RMS [m]: {np.sqrt(np.mean(good_resid**2)):.4f}')
print(f'Sample residual standard deviation [m]: {np.std(good_resid, ddof=1):.4f}')
print(f'Robust MAD-derived residual scale [m]: '
      f'{1.4826*np.median(np.abs(good_resid-np.median(good_resid))):.4f}')
print(f'Unit-weight sigma-hat (dimensionless): {sigma_hat0:.4f}')
print(f'Max active |v/σ_i|: {np.max(np.abs(vpost[active]/sigma_i[active])):.3f}')
assert dof > 0

fig, ax = plt.subplots(2, 2, figsize=(11, 8))
ax[0, 0].scatter(obs_times_h[active], vpost[active], s=9)
if np.any(~active):
    ax[0, 0].scatter(obs_times_h[~active], vpost[~active],
                     marker='x', color='tab:red', label='Rejected rows')
    ax[0, 0].legend()
ax[0, 0].set(xlabel='Hours', ylabel='Postfit residual [m]', title='Time structure')

ax[0, 1].scatter(np.rad2deg(el_rad[active]), vpost[active], s=9)
ax[0, 1].set(xlabel='Elevation [degrees]', ylabel='Postfit residual [m]',
             title='Elevation dependence')

ax[1, 0].hist(good_resid, bins=28, edgecolor='white')
ax[1, 0].set(xlabel='Postfit residual [m]', ylabel='Count', title='Distribution')

normalized_v = vpost[active]/sigma_i[active]
ax[1, 1].scatter(np.rad2deg(el_rad[active]), normalized_v, s=9)
ax[1, 1].axhline(+4, ls='--', color='tab:red', label='rejection ±4σ')
ax[1, 1].axhline(-4, ls='--', color='tab:red')
ax[1, 1].set(xlabel='Elevation [degrees]', ylabel='v / a priori σ',
             title='Standardized residuals')
ax[1, 1].legend()
for a in ax.flat: a.grid(alpha=.2)
plt.tight_layout()
plt.show()


### What can we infer from the residuals?

- **RMS** tells us the typical *remaining code-range disagreement in metres*,
  after estimating XYZ, clocks and wet delays. It is not a position error.
- **Standard deviation** describes scatter about the residual mean; RMS
  also includes any nonzero mean.
- **$\hat\sigma_0$ near 1** means residual scatter broadly agrees with our
  assumed $5/\sin E$-metre model. Much smaller than 1 suggests our a priori
  noise model is conservative *or* the parameterization is flexible;
  larger than 1 suggests optimistic weights or unmodeled effects.
- **Patterns versus elevation or time** can indicate unmodeled multipath,
  atmosphere, equipment changes or modeling errors. Residual patterns
  cannot by themselves identify the cause uniquely.
- **No rejected rows** is a perfectly legitimate result if no residual
  exceeds the four-sigma threshold.

Because a new receiver-clock parameter is estimated for **every epoch**,
clock errors largely disappear into these parameters. Similarly, slow wet
errors can be absorbed by our piecewise ZWD. We must not interpret small
post-fit residuals as proof of accurate absolute station coordinates.


## 11. What did we estimate? Receiver clocks and **ZWD knots**

Extract the parameters using the exact slices shown earlier:

$$\hat{\mathbf r}_{marker}=\hat\theta_{0:3},\qquad
\hat b_k=\hat\theta_{3+k},\qquad
\widehat{ZWD}_j=\hat\theta_{3+N_e+j}.$$

The receiver-clock parameters $b_k$ are in **metres**; divide by $c$ to
display seconds or microseconds. The ZWD unknowns live at **two-hour
knots**. For a particular observation, get its estimated wet delay by
interpolating **these adjusted knot values**, not by reusing the VMF3
prior. The VMF3 a priori ZWD is only our initial guess. The current
unconstrained least-squares model does **not** force ZWD to be positive.
An unphysical negative fitted knot is a reason to inspect geometry,
model assumptions and parameter correlations—not a negative atmosphere.


In [ ]:
estimated_xyz = x[:3]
receiver_clocks_m = x[3:3+n_epochs]
zwd_knots_m = x[3+n_epochs:]

# One real timestamp per kept clock epoch (several satellites share it).
clock_times_h = np.full(n_epochs, np.nan)
for obs, h in zip(raw_obs, obs_times_h):
    clock_times_h[obs['nepoch']] = h
assert np.all(np.isfinite(clock_times_h))

zwd_estimated_at_obs = np.array([
    sum(weight*zwd_knots_m[index] for index, weight in o['zwd_basis'])
    for o in raw_obs
])

fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(clock_times_h, receiver_clocks_m/gs.C*1e6, '.-',
           label='Estimated receiver-clock offset')
ax[0].set(xlabel='Hours', ylabel='Estimated receiver clock [µs]',
          title='One clock unknown for each accepted epoch')

zwd_knot_times = np.arange(n_zwd)*ZWD_EVERY_HOURS
ax[1].plot(zwd_knot_times, zwd_knots_m, 'o-', label='Adjusted ZWD knot values')
ax[1].scatter(obs_times_h, zwd_estimated_at_obs, s=8,
              label='Interpolated ZWD at observation times')
ax[1].scatter(obs_times_h, [o['dT']['zwd'] for o in raw_obs], s=7,
              alpha=.4, label='VMF3 a priori ZWD (not adjusted)')
ax[1].set(xlabel='Hours', ylabel='Zenith wet delay [m]',
          title='Two-hour knot model: smooth between consecutive knots')
for a in ax:
    a.grid(alpha=.25)
    a.legend()
plt.tight_layout()
plt.show()

print('Adjusted marker XYZ [m]:', estimated_xyz)
print('First five adjusted wet-delay knots [m]:', zwd_knots_m[:5])
print('Wet-delay range across observations [m]:',
      zwd_estimated_at_obs.min(), zwd_estimated_at_obs.max())


## 12. Formal uncertainties (not a proof of absolute accuracy)

For this nonlinear adjustment, use the **final** Jacobian and active weights.
The standard *linearized* covariance expression is

$$\mathrm{Cov}(\hat\theta)\approx
\hat\sigma_0^2(\mathbf A^T\mathbf W\mathbf A)^{-1}.$$

This matrix contains variances on the diagonal and cross-correlations
between the estimated XYZ, clocks and wet-delay knots. We compute it via
an SVD of the **weighted Jacobian**, avoiding a direct normal-matrix
inverse. This step assumes the linearized model, the estimated weights
and correct model rank; the resulting **formal sigmas** are not equivalent
to independent coordinate accuracy against SINEX.

Large correlations, particularly between station **height**, receiver
clocks, and slowly varying ZWD, remind us why satellite geometry and
observation coverage matter.


In [ ]:
sqrt_w = np.sqrt(w_final[active])
Aw_final = A_final[active]*sqrt_w[:, None]
U, singular, Vt = np.linalg.svd(Aw_final, full_matrices=False)
if len(singular) != n_params or singular[-1] <= 0:
    raise ValueError('Final weighted Jacobian lacks full column rank')

# A = U S V^T. Then (A^T A)^{-1} = V diag(1/s²) V^T.
cofactor = (Vt.T / singular**2) @ Vt
covariance = sigma_hat0**2 * cofactor
formal_sigma = np.sqrt(np.maximum(0.0, np.diag(covariance)))

print('Formal σX, σY, σZ [m]:', formal_sigma[:3])
print('Formal σ of receiver clocks [m], first five:',
      formal_sigma[3:3+min(5,n_epochs)])
print('Formal σ of ZWD knots [m], first five:',
      formal_sigma[3+n_epochs:3+n_epochs+min(5,n_zwd)])
print('Weighted-Jacobian condition number:', singular[0]/singular[-1])

xyz_cov = covariance[:3, :3]
xyz_correlations = xyz_cov / np.outer(formal_sigma[:3], formal_sigma[:3])
print('XYZ parameter correlation matrix:')
print(np.round(xyz_correlations, 3))


## 13. Optional classroom experiment: inject **one known bad code**

A gross error should be identified by the final **standardized residual**
when it cannot be absorbed by the estimated parameters. This optional
experiment adds **100 metres** to one *copy* of an actual DYNG code value
at the highest elevation. This is **not** a naturally occurring RINEX error;
the raw file and original adjustment remain untouched.

Rerun our same iteration to inspect which row is rejected. Depending on
satellite geometry, the error may partly affect the clock and other
parameters; a residual check is necessary. A failed re-adjustment is
reported as such, not silently ignored.


In [ ]:
bad_index = int(np.argmax(el_rad))
bad_observed = observed_m.copy()
bad_observed[bad_index] += 100.0

try:
    x_bad, residual_bad, weights_bad, _, sigma_bad, history_bad = (
        iterate_code_positioning(bad_observed, raw_obs, x0, sigma_i,
                                 n_epochs, n_zwd))
except (ValueError, RuntimeError) as exc:
    print('Gross-error experiment could not be adjusted:', exc)
else:
    print('Injected +100 m code error at observation index:', bad_index)
    print('Real satellite:', raw_obs[bad_index]['sat'])
    print('Injected row rejected?', weights_bad[bad_index] == 0)
    print('Number of rows rejected in experiment:', np.count_nonzero(weights_bad == 0))
    print('Post-fit residual at injected index [m]:', residual_bad[bad_index])
    print('Original station XYZ vs perturbed adjustment [m]:',
          x_bad[:3]-estimated_xyz)


## 14. What to remember before we move to carrier phase

1. **One code observation → one design row.** Our unknowns are XYZ, one
   receiver clock per accepted epoch, and shared piecewise ZWD knot values.
2. **Each row uses only one clock parameter and two ZWD parameters.** A
   ZWD model with one independent unknown per satellite observation would
   absorb the code data instead of constraining a coherent atmosphere.
3. **Whiten the design and misclosure** using the square roots of the
   weights; solve with `lstsq`, not a normal-matrix inverse.
4. **Rebuild the geometry after updating XYZ.** Test convergence on both
   XYZ and the clock/wet-delay parameter increments.
5. **Use the final nonlinear residuals**, not the initial misclosures or
   the previous iteration's residuals, for quality diagnostics.
6. Residual RMS, $\hat\sigma_0$ and formal coordinate sigmas answer
   **different questions**. Independent SINEX coordinates are still useful
   for checking the estimated position.

The next carrier-phase lesson will extend the same design-matrix idea
with **one new phase-ambiguity parameter per continuous satellite arc**;
the code-only clock, geometry and ZWD columns remain conceptually similar.
